# Notebook 3 — A model that steps through time, and a posterior that goes wrong

This is the notebook that looks most like real work. Two things here:

**Part A — `pytensor.scan`.** Most interesting models are not one formula. They are a loop:
today's state depends on yesterday's state. You cannot write that with a plain Python `for`
loop inside a PyMC model, because NUTS needs to differentiate the whole thing.
`pytensor.scan` is the loop that can be differentiated.

**Part B — what an unidentifiable posterior looks like.** We will build the *same* model
twice with different data, and the second one will fail badly: R-hat well above 1, an
effective sample size in the tens, hundreds of divergences. Learning to recognise this on a toy model, where
you know the truth, is much easier than meeting it for the first time on real data.

**The example:** one box holding some amount of stuff `C`. Each day, a fixed amount comes in,
and a fixed *fraction* of what is in the box leaves.

```
C_today = C_yesterday + input - k * C_yesterday
```

Two unknowns: `input` (how much arrives per day) and `k` (the fraction lost per day).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import pytensor
import pytensor.tensor as pt
import arviz as az

print("pymc", pm.__version__, "| pytensor", pytensor.__version__, "| arviz", az.__version__)

## Part A — The forward model

### First in plain NumPy, so you can see what it does

In [ ]:
rng = np.random.default_rng(11)

TRUE_K = 0.02      # fraction lost per day
TRUE_IN = 0.5      # amount added per day
NDAYS = 120

def run_numpy(C0, k, inp, ndays):
    C = np.empty(ndays)
    c = C0
    for t in range(ndays):
        c = c + inp - k * c
        C[t] = c
    return C

C0_high = 100.0
truth = run_numpy(C0_high, TRUE_K, TRUE_IN, NDAYS)
obs = truth + rng.normal(0, 1.5, NDAYS)   # noisy daily measurements

plt.figure(figsize=(7, 3.5))
plt.plot(truth, lw=2, label="true C")
plt.scatter(np.arange(NDAYS), obs, s=8, color="C1", alpha=.6, label="noisy observations")
plt.axhline(TRUE_IN / TRUE_K, ls="--", color="grey", label="equilibrium = input / k")
plt.xlabel("day"); plt.ylabel("C"); plt.legend()
plt.title("Starting high, decaying towards equilibrium")
plt.show()

Notice the dashed line. If you leave this model running forever it settles at
`input / k`. That fact is going to matter a lot in Part B.

### Now the same loop in `pytensor.scan`

`scan` takes:

- `fn` — what to do on one step. It receives the previous value, and returns the new one.
- `outputs_info` — the starting value.
- `non_sequences` — things that stay the same on every step (your parameters).
- `n_steps` — how many steps.

**One practical tip that matters:** pass your parameters through `non_sequences`, not by
grabbing them from the surrounding Python scope. Both *work*, but going through
`non_sequences` makes the compiled gradient far faster. On a real model this can be the
difference between an hour and a day.

In [ ]:
def one_step(C_prev, k, inp):
    """One day. C_prev is yesterday's value; k and inp are the parameters."""
    return C_prev + inp - k * C_prev


def build_scan(C0, k, inp, ndays):
    seq, _updates = pytensor.scan(
        fn=one_step,
        outputs_info=[pt.as_tensor_variable(np.float64(C0))],
        non_sequences=[k, inp],
        n_steps=ndays,
    )
    return seq

### Check it against NumPy before you trust it

Always do this. A scan that is subtly wrong will still sample happily and give you a
confident, wrong answer.

In [ ]:
k_sym = pt.dscalar("k")
in_sym = pt.dscalar("inp")
seq_sym = build_scan(C0_high, k_sym, in_sym, NDAYS)
f = pytensor.function([k_sym, in_sym], seq_sym)

diff = np.abs(f(TRUE_K, TRUE_IN) - truth).max()
print("largest difference vs the NumPy loop:", diff)
assert diff < 1e-10, "scan does not match the plain loop!"
print("scan matches. good.")

In [ ]:
# And check the gradient exists. If this errors, NUTS cannot be used.
g = pytensor.grad(seq_sym.sum(), k_sym)
grad_fn = pytensor.function([k_sym, in_sym], g)
print("d(sum of C)/dk at the true values:", grad_fn(TRUE_K, TRUE_IN))

## The good case: data that identifies both parameters

The observations start high and decay. The *shape* of that decay tells you `k`,
and the level it settles at tells you `input / k`. Two facts, two unknowns. Solvable.

In [ ]:
with pm.Model() as good_model:
    k = pm.Uniform("k", 0.001, 0.2)
    inp = pm.Uniform("inp", 0.0, 3.0)
    sigma = pm.HalfNormal("sigma", 3.0)

    C = build_scan(C0_high, k, inp, NDAYS)
    pm.Deterministic("C", C)
    pm.Normal("obs", mu=C, sigma=sigma, observed=obs)

    idata_good = pm.sample(1000, tune=1000, chains=4, cores=1, random_seed=21)

print(az.summary(idata_good, var_names=["k", "inp", "sigma"]))
print()
print(f"true k = {TRUE_K}, true inp = {TRUE_IN}, true sigma = 1.5")
print("divergences:", int(idata_good.sample_stats["diverging"].sum()))

In [ ]:
kg = np.asarray(idata_good.posterior["k"]).ravel()
ig = np.asarray(idata_good.posterior["inp"]).ravel()

plt.figure(figsize=(4.5, 4))
plt.scatter(kg, ig, s=3, alpha=.2)
plt.scatter([TRUE_K], [TRUE_IN], color="red", s=60, marker="x", zorder=5, label="truth")
plt.xlabel("k"); plt.ylabel("input")
plt.title(f"a blob, not a streak. r = {np.corrcoef(kg, ig)[0,1]:.2f}")
plt.legend(); plt.show()

### Uncertainty in the parameters becomes uncertainty in the whole trajectory

Because we saved `C` with `pm.Deterministic`, every posterior draw carries a full
120-day trajectory. Plotting the spread of those trajectories shows how parameter
uncertainty travels through the model into the thing you actually care about.

In [ ]:
C_draws = np.asarray(idata_good.posterior["C"]).reshape(-1, NDAYS)
lo, hi = np.percentile(C_draws, [2.5, 97.5], axis=0)

plt.figure(figsize=(7, 3.5))
plt.fill_between(np.arange(NDAYS), lo, hi, alpha=.35, label="95% of trajectories")
plt.plot(C_draws.mean(axis=0), lw=2, label="posterior mean")
plt.plot(truth, "--", color="k", lw=1, label="truth")
plt.scatter(np.arange(NDAYS), obs, s=6, color="C1", alpha=.4, label="observations")
plt.xlabel("day"); plt.ylabel("C"); plt.legend(); plt.show()

## Part B — The bad case: the same model, data that cannot tell the parameters apart

Now change one thing. Start the box **already at equilibrium** (`C0 = input / k`).
The trajectory is then flat. A flat line at height 25 tells you that `input / k = 25`,
and nothing else. `k = 0.02, input = 0.5` fits. So does `k = 0.1, input = 2.5`.
So does `k = 0.004, input = 0.1`.

This is **equifinality**: many different parameter combinations produce the same output,
so the data cannot choose between them. The model is fine. The data just is not informative
about these two things separately.

In [ ]:
NDAYS_B = 90
C0_eq = TRUE_IN / TRUE_K          # start exactly at equilibrium
truth_b = run_numpy(C0_eq, TRUE_K, TRUE_IN, NDAYS_B)
obs_b = truth_b + rng.normal(0, 1.0, NDAYS_B)

plt.figure(figsize=(7, 3))
plt.plot(truth_b, lw=2)
plt.scatter(np.arange(NDAYS_B), obs_b, s=8, color="C1", alpha=.6)
plt.xlabel("day"); plt.ylabel("C")
plt.title("A flat line. Informative about input/k, and nothing else.")
plt.show()

In [ ]:
with pm.Model() as bad_model:
    k = pm.Uniform("k", 0.001, 0.2)
    inp = pm.Uniform("inp", 0.0, 3.0)
    sigma = pm.HalfNormal("sigma", 3.0)

    C = build_scan(C0_eq, k, inp, NDAYS_B)
    pm.Normal("obs", mu=C, sigma=sigma, observed=obs_b)

    idata_bad = pm.sample(1000, tune=1000, chains=4, cores=1, random_seed=22)

Read the warnings PyMC just printed. It is telling you exactly what is wrong.
Now the summary table.

In [ ]:
print(az.summary(idata_bad, var_names=["k", "inp", "sigma"]))
print()
print(f"true k = {TRUE_K}, true inp = {TRUE_IN}")
print("divergences:", int(idata_bad.sample_stats["diverging"].sum()))

Everything that can be bad, is bad:

- **`r_hat` far above 1.01** — the four chains ended up in different places. They disagree.
- **`ess_bulk` of single digits** — 4000 draws are worth about 7 independent ones.
- **hundreds of divergences** — the sampler kept falling off the ridge.
- **the means are nowhere near the truth**

None of these numbers should be reported. The run is not usable.

### The picture: a ridge, with divergences sitting on it

`az.plot_pair` can mark divergences for you in ArviZ 0.x, but the argument changed in 1.x.
Doing it by hand works everywhere, and it teaches you where the divergence flags live —
in `sample_stats`, one True/False per draw.

In [ ]:
kb = np.asarray(idata_bad.posterior["k"]).ravel()
ib = np.asarray(idata_bad.posterior["inp"]).ravel()
div = np.asarray(idata_bad.sample_stats["diverging"]).ravel().astype(bool)

plt.figure(figsize=(5.5, 4.5))
plt.scatter(kb[~div], ib[~div], s=4, alpha=.25, label="ok draws")
plt.scatter(kb[div], ib[div], s=14, color="red", alpha=.7, label="divergences")

kk = np.linspace(0.001, 0.2, 200)
plt.plot(kk, 25 * kk, "k--", lw=1, label="input = 25 * k  (the ridge)")

plt.scatter([TRUE_K], [TRUE_IN], color="lime", s=90, marker="x", zorder=5, label="truth")
plt.xlim(0, 0.2); plt.ylim(0, 3)
plt.xlabel("k"); plt.ylabel("input")
plt.title(f"correlation = {np.corrcoef(kb, ib)[0,1]:.4f}")
plt.legend(fontsize=8); plt.show()

### The key insight: something *is* well determined

The two parameters are hopeless on their own. But their **ratio** is pinned down
beautifully — it is the equilibrium level, and the flat data measures that very precisely.

So the honest statement is not "the model failed". It is: *"this data constrains
`input / k`, and says almost nothing about either one separately."*

In [ ]:
ratio = ib / kb

print(f"k       : mean {kb.mean():.4f}   sd {kb.std():.4f}   <- useless")
print(f"input   : mean {ib.mean():.4f}   sd {ib.std():.4f}   <- useless")
print(f"input/k : mean {ratio.mean():.4f}   sd {ratio.std():.4f}   <- tight!")
print(f"true input/k = {TRUE_IN / TRUE_K}")

plt.figure(figsize=(6, 3))
plt.hist(ratio, bins=50)
plt.axvline(TRUE_IN / TRUE_K, color="red", lw=2, label="truth")
plt.xlabel("input / k"); plt.legend()
plt.title("The combination the data can actually see")
plt.show()

### Two more diagnostics worth knowing

**Chains disagreeing.** Plot each chain's draws separately. In the bad run they sit in
different places; in the good run they overlap.

**BFMI** (Bayesian Fraction of Missing Information) is a number per chain that says whether
the sampler is exploring properly. Below about 0.3 is a warning sign. `az.plot_energy`
draws the same idea as a picture: the two curves it draws should sit on top of each other.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for i in range(idata_good.posterior.sizes["chain"]):
    ax[0].scatter(np.asarray(idata_good.posterior["k"][i]),
                  np.asarray(idata_good.posterior["inp"][i]), s=3, alpha=.3, label=f"chain {i}")
ax[0].set_title("good run: chains on top of each other")
ax[0].set_xlabel("k"); ax[0].set_ylabel("input"); ax[0].legend(fontsize=7)

for i in range(idata_bad.posterior.sizes["chain"]):
    ax[1].scatter(np.asarray(idata_bad.posterior["k"][i]),
                  np.asarray(idata_bad.posterior["inp"][i]), s=3, alpha=.3, label=f"chain {i}")
ax[1].set_title("bad run: each chain parked somewhere different")
ax[1].set_xlabel("k"); ax[1].set_ylabel("input"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
def bfmi_values(idata):
    # az.bfmi returns a plain array in ArviZ 0.x and a labelled object in 1.x
    out = az.bfmi(idata)
    return np.round(np.asarray(getattr(out, "energy", out)).ravel(), 3)

print("BFMI per chain, good run:", bfmi_values(idata_good))
print("BFMI per chain, bad run :", bfmi_values(idata_bad))
print("(below about 0.3 is a warning sign)")

az.plot_energy(idata_bad)
plt.show()

## Three ways to deal with a ridge like this

1. **Get better data.** Here, observing the decay from a non-equilibrium start fixes it
   completely — that was the good case. Different data, same model.
2. **Sample the combination instead.** Put a prior on `ratio = input / k` and on `k`,
   and compute `input = ratio * k`. The sampler then moves along nice directions instead
   of crawling along a diagonal.
3. **Say so honestly.** Sometimes the data really cannot separate two things, and the right
   result is to report the combination that *is* determined, and say the individual
   parameters are not.

Try option 2 below.

In [ ]:
with pm.Model() as fixed_model:
    ratio_p = pm.Uniform("ratio", 1.0, 200.0)     # the equilibrium level
    k = pm.Uniform("k", 0.001, 0.2)
    inp = pm.Deterministic("inp", ratio_p * k)    # calculated, not sampled
    sigma = pm.HalfNormal("sigma", 3.0)

    C = build_scan(C0_eq, k, inp, NDAYS_B)
    pm.Normal("obs", mu=C, sigma=sigma, observed=obs_b)

    idata_fix = pm.sample(1000, tune=1000, chains=4, cores=1, random_seed=23)

print(az.summary(idata_fix, var_names=["ratio", "k", "inp", "sigma"]))
print()
print("divergences:", int(idata_fix.sample_stats["diverging"].sum()))
print(f"true ratio = {TRUE_IN / TRUE_K}")

`ratio` should now have a good `r_hat` and a much healthier `ess_bulk` — because it is the
thing the data can see. `k` and `inp` are still badly determined, and that is correct: the
data genuinely does not know them.

You will probably still see divergences. That is honest too — the model still contains a
direction the data cannot see, so the sampler still has a hard region to cross. What
changed is that the number you can actually report, `ratio`, now converges, and the table
tells you clearly which parameters are trustworthy and which are not. Before, the whole
run was unusable.

## Your turn — exercises

1. **Find the tipping point.** In the bad case, start at `C0 = 30` instead of exactly 25.
   Then try 26, 27, 40. How far from equilibrium does the start have to be before the
   parameters separate?
2. **Shorten the good run.** Use only the first 15 days of the good dataset. The decay is
   barely visible. Watch the correlation between `k` and `input` climb.
3. **Change the noise.** Set the observation noise to 8 instead of 1.5 in the good case.
   How much wider does the posterior get, and does the correlation come back?
4. **Two boxes.** Change `one_step` so that the stuff leaving box 1 arrives in box 2, and
   box 2 also loses a fraction each day. `outputs_info` now needs two starting values and
   `fn` returns two things. Observe only box 2. Can you still recover box 1's loss rate?
5. **Speed test.** Rewrite `build_scan` so that `k` and `inp` are captured from the
   surrounding Python scope instead of being passed through `non_sequences`. Time
   `pytensor.grad` compilation and evaluation both ways.
6. **Add a parameter the model never uses** — say `dead = pm.Uniform("dead", 0, 1)` that
   appears nowhere in `one_step`. Sample. Its posterior will be exactly its prior.
   That is what a parameter the likelihood cannot see looks like.